# Fixed SARIMA forecasting system

Cleaned from your training notebook. Rebuilds everything from raw metadata and readings, with no dependency on other notebook kernels. Run time is a few minutes.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

search = [Path.cwd(), *Path.cwd().parents]
ROOT = next(candidate for parent in search for candidate in (parent, parent / 'deliver_ready')
            if (candidate / 'main.py').exists() and (candidate / 'src/workload/data.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))
from workload.data import load_data, split, validate_predictions

# Set VM_DATA_DIR, or change this variable to the directory with the two raw files.
candidates = [ROOT / 'challenge_data', ROOT.parent / 'forecasting-challenge/challenge_data']
DATA_DIR = Path(os.environ['VM_DATA_DIR']) if 'VM_DATA_DIR' in os.environ else next(
    (p for p in candidates if (p / 'vm_cpu_readings.csv.gz').exists() or (p / 'vm_cpu_readings.csv').exists()), candidates[0])
data = load_data(DATA_DIR)
n, future_times = split(data)
RESULTS = ROOT / 'src/results'
OUT = ROOT / 'notebooks/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'{n} training timestamps; {len(future_times)} forecast timestamps; raw files: {DATA_DIR}')


## Recipe and validation

Fit group-level SARIMA `(1,0,1) × (2,1,0,12/24)`. Compute `p12 − line(p12) + line(p24)`, bound to [0,100], then merge by VM counts. The period-12 component orders were assumed to match the saved period-24 model because the original period-12 fit definition was missing. Validate at origins 62 and 86; final origin 110 predicts 24 timestamps.

**My observations and explanation:**

_Add your notes here._

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('submission_entry', ROOT / 'main.py')
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
predictions, manifest = entry.main(['--data-dir', str(DATA_DIR), '--results-dir', str(RESULTS), '--evaluate'])
validate_predictions(predictions, future_times)
display(predictions)

## Main forecasts and all nine histories

Dotted observed holdout values are shown retrospectively after predictions have been written; they are not model inputs.

**My observations and explanation:**

_Add your notes here._

In [ ]:
display(Image(filename=str(RESULTS / 'figures/forecast.png')))
display(Image(filename=str(RESULTS / 'figures/groups_full_history.png')))

## What was removed

No hidden `fit_12` globals, merged-file reads, metadata CPU summary features, CPU terciles from holdout data, or `109 * 3600` cutoff. Relative timestamps are seconds with a 300-second cadence. Earlier public-holdout inspection remains a limitation even though this implementation fits only training prefixes.